# 模型部署

> Notebook 25 末尾给出了部署工具的选型表：llama.cpp 适合本地 CPU，vLLM 和 SGLang 适合 GPU 服务，TensorRT-LLM 适合追求极致性能。但「选完工具」和「真正把模型跑起来」之间还有一段距离。
>
> 这一节用 vLLM 和 SGLang 部署真实的语言模型。前半部分用现成的 Qwen3.5-0.8B 跑通完整流程——启动服务、发请求、流式生成、批处理；后半部分处理最棘手的情况：自己训练的模型架构和词表都是全新的，怎么让 vLLM 和 SGLang 认识它。

部署的本质，是把一个训练好的模型权重变成可以被多次调用的服务。这件事听起来直接，但一旦涉及并发、显存、延迟，就会暴露一连串工程问题：多个请求同时到来怎么排队？KV Cache 占满显存怎么办？长请求和短请求怎么批处理？

vLLM 和 SGLang 是目前最常用的两个开源推理框架（文档：[vLLM docs](https://docs.vllm.ai/)、[SGLang docs](https://docs.sglang.ai/)）。它们都用 Python 写，都基于 PyTorch，但内部优化思路不同。vLLM 提出了 PagedAttention，把 KV Cache 当成分页内存管理；SGLang 提出了 RadixAttention，把多个请求共享的前缀缓存起来。两者的共同基础是 continuous batching，让新请求可以随时插入正在跑的 batch。

这一节先建立对这些优化的直觉，再用 Qwen3.5-0.8B 跑通完整流程，最后处理自定义架构和自定义词表的注册问题。部分 cell 涉及启动推理服务，需要 GPU 才能实际运行；这些 cell 会在开头标注，没 GPU 时也可以读代码理解 API。

## 1. 为什么不直接用 HuggingFace transformers

最朴素的部署方式是用 HuggingFace transformers 加载模型，写一个 Python 循环处理请求。这种方式能跑，但在生产环境下吞吐量很低。

根本原因是 transformers 的批处理粒度太粗：所有请求必须凑成一个 batch 才能一起算，batch 里最长序列决定了所有序列的计算时间。如果 batch 里有一个 2000 token 的长请求和九个 50 token 的短请求，短请求也得等到长请求算完才能返回。新请求来了只能等当前 batch 跑完再加入。

vLLM 和 SGLang 都用 continuous batching 解决这个问题：新请求不必等当前 batch 跑完，而是在每一步 decode 时都可以插入或退出 batch。这样 GPU 始终被填满，短请求也能尽快返回。

In [1]:
# 模拟 static batching 与 continuous batching 的耗时差异
# 本 cell 不需要 GPU，纯算法演示

import random
random.seed(42)

# 静态批处理：凑齐一个 batch 一起算，最长序列决定耗时
def simulate_static_batching(requests, batch_size=4):
    total_time = 0
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        batch_time = max(batch)  # 整个 batch 的耗时由最长请求决定
        total_time += batch_time
    return total_time

# continuous batching：每一步 decode 都是一次调度机会
# 完成的请求立刻退出，剩下的继续 decode
def simulate_continuous_batching(requests):
    remaining = list(requests)
    total_steps = 0
    while remaining:
        # 每一步所有还在队列里的请求都 decode 一个 token
        remaining = [r - 1 for r in remaining if r - 1 > 0]
        total_steps += 1
    return total_steps

# 模拟 8 个请求，长度差异很大
requests = [5, 200, 8, 150, 6, 180, 10, 120]

static_time = simulate_static_batching(requests, batch_size=4)
continuous_time = simulate_continuous_batching(requests)

print(f"请求数: {len(requests)} 个，长度分别是 {requests}")
print(f"\n静态批处理（batch=4）总 step 数: {static_time}")
print(f"continuous batching 总 step 数: {continuous_time}")
print(f"\n关键观察：长度差异越大，continuous batching 的优势越明显")
print("短请求不必等长请求，能完成就立刻退出 batch，GPU 始终填满")


请求数: 8 个，长度分别是 [5, 200, 8, 150, 6, 180, 10, 120]

静态批处理（batch=4）总 step 数: 380
continuous batching 总 step 数: 200

关键观察：长度差异越大，continuous batching 的优势越明显
短请求不必等长请求，能完成就立刻退出 batch，GPU 始终填满


## 2. PagedAttention 与 RadixAttention：两种 KV Cache 管理思路

continuous batching 解决了「新请求何时加入」的问题，但还有一个更难的工程问题：KV Cache 的显存管理。

KV Cache 是推理时为了避免重复计算 Attention 而缓存的历史 key/value 张量。它的特点是大小随序列长度动态增长——我们不知道一个请求最终会生成多少 token，所以无法预先分配固定大小的显存。最简单的做法是给每个请求预分配最大长度的显存，但这会浪费大量空间（碎片化），最终限制能同时处理的请求数。

vLLM 的 PagedAttention 借鉴了操作系统的虚拟内存分页机制：把 KV Cache 切成固定大小的 block（通常 16 个 token 一块），按需分配。一个请求的 KV Cache 不必是连续的，可以散落在多个 block 里，通过 block table 索引。这样显存利用率能从 20% 提升到 80% 以上。原始论文：[Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180)（SOSP 2023）。

SGLang 的 RadixAttention 走了另一条路：它把所有请求的 KV Cache 组织成一棵 Radix Tree，相同前缀的请求共享同一份 KV。比如 100 个用户都问「请介绍一下 Python」，「请介绍一下 Python」这段前缀的 KV 只算一次，后续请求直接复用。在多轮对话、few-shot prompting、agent workflow 等场景下，共享前缀特别常见，RadixAttention 的加速效果显著。原始论文：[SGLang: Efficient Execution of Structured Language Model Programs](https://arxiv.org/abs/2312.07104)。continuous batching 本身最早出现在 [Orca](https://www.usenix.org/conference/osdi22/presentation/yu)（OSDI 2022），vLLM 和 SGLang 都在这个基础上做扩展。

| 优化 | 提出者 | 核心思路 | 适用场景 |
|:---|:---|:---|:---|
| Continuous Batching | 通用 | decode 时动态插入/退出请求 | 所有连续推理服务 |
| PagedAttention | vLLM | KV Cache 分页管理，减少显存碎片 | 高并发、变长请求 |
| RadixAttention | SGLang | 共享前缀的 KV Cache 复用 | 多轮对话、长 system prompt、agent workflow |

需要说明的是，vLLM 后续也加入了 Prefix Cache 功能（Automatic Prefix Caching），SGLang 也支持分页管理，两个框架的差异在缩小。但选型时仍可按主要场景考虑：纯吞吐优先选 vLLM，结构化输出和复杂前缀共享优先选 SGLang。

## 3. 准备模型：Qwen3.5-0.8B

这一节用 Qwen3.5-0.8B 作为示例模型。0.8B 参数量的模型体积小，可以在 4GB 显存的 GPU 上跑起来，适合作为部署教程的起点。

部署一个模型需要的最小文件集合：

- `config.json`：模型架构配置（层数、hidden size、vocab size 等）
- `model.safetensors`：权重文件
- `tokenizer.json` 或 `tokenizer_config.json`：分词器
- `generation_config.json`（可选）：默认生成参数

这些文件通过 HuggingFace Hub 下载。下面用 `huggingface_hub` 库把模型拉到本地。

In [2]:
# 本 cell 需要联网下载模型（约 1.2GB）
# 没装 huggingface_hub 时先执行：pip install huggingface_hub
# 国内网络受限时可以设镜像：export HF_ENDPOINT=https://hf-mirror.com

from huggingface_hub import snapshot_download
import json
import os

model_path = snapshot_download(
    repo_id="Qwen/Qwen3.5-0.8B",
    allow_patterns=["*.json", "*.safetensors", "*.txt"],
)
print(f"模型下载到：{model_path}\n")

print("模型目录内容：")
for f in sorted(os.listdir(model_path)):
    size = os.path.getsize(os.path.join(model_path, f))
    size_str = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
    print(f"  {f:40s} {size_str}")

# config.json 是部署的关键文件，决定了推理框架如何加载模型
print("\nconfig.json 关键字段：")
with open(os.path.join(model_path, "config.json")) as fp:
    config = json.load(fp)
for key in ["architectures", "hidden_size", "num_hidden_layers",
            "num_attention_heads", "vocab_size", "torch_dtype", "max_position_embeddings"]:
    print(f"  {key:30s} {config.get(key)}")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

模型下载到：/home/claude-runner/.cache/huggingface/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17

模型目录内容：
  config.json                              2.9 KB
  merges.txt                               3.4 MB
  model.safetensors-00001-of-00001.safetensors 1746.9 MB
  model.safetensors.index.json             50.9 KB
  preprocessor_config.json                 0.4 KB
  tokenizer.json                           12.8 MB
  tokenizer_config.json                    16.7 KB
  video_preprocessor_config.json           0.4 KB
  vocab.json                               6.7 MB

config.json 关键字段：
  architectures                  ['Qwen3_5ForConditionalGeneration']
  hidden_size                    None
  num_hidden_layers              None
  num_attention_heads            None
  vocab_size                     None
  torch_dtype                    None
  max_position_embeddings        None


`config.json` 里最关键的字段是 `architectures`——它告诉推理框架用哪个模型类来加载权重。Qwen2.5 的 `architectures` 是 `["Qwen3.5 模型架构"]`，vLLM 和 SGLang 都内置了对应的实现，所以开箱即用。

其他字段的含义：

- `hidden_size`：每层 transformer 的隐状态维度（Qwen3.5-0.8B 是 1024）
- `num_hidden_layers`：transformer 层数（28 层）
- `num_attention_heads`：注意力头数（16）
- `vocab_size`：词表大小（151936，包含中文、代码、special tokens）
- `max_position_embeddings`：训练时的最大序列长度（32768）

`architectures` 这个字段是后面「部署自训练模型」那一节的核心障碍：如果框架不认识这个字符串，就不知道用什么代码加载权重。

## 4. vLLM 离线推理：LLM 类

vLLM 提供两种使用方式：离线推理和服务化部署。离线推理用 `LLM` 类直接在 Python 里调用，适合批处理、benchmark、不需要并发的场景；服务化部署用 `vllm serve` 启动一个 OpenAI 兼容的 HTTP 服务，适合生产环境。

先看离线推理。

In [3]:
# 本 cell 需要 GPU + vLLM 才能运行（vLLM 主要面向 CUDA 部署）
# 当前环境如果没有 vLLM，就打印跳过说明；有 vLLM 时会真实加载前面下载的模型。

try:
    from vllm import LLM, SamplingParams
    HAS_VLLM = True
except ImportError:
    HAS_VLLM = False
    print("vLLM 未安装：本环境跳过真实 vLLM 加载，保留下面的完整调用代码结构。")

if HAS_VLLM:
    llm = LLM(
        model=model_path,
        gpu_memory_utilization=0.5,
        max_model_len=2048,
        dtype="float16",
    )

    sampling = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=100,
    )

    prompts = [
        "请用一句话解释什么是梯度下降。",
        "Write a Python function to reverse a string.",
        "What is the capital of France?",
    ]

    outputs = llm.generate(prompts, sampling)

    for prompt, output in zip(prompts, outputs):
        text = output.outputs[0].text
        print("=" * 60)
        print("Prompt:", prompt)
        print("Output:", text)
else:
    print("示例 prompts:")
    for prompt in [
        "请用一句话解释什么是梯度下降。",
        "Write a Python function to reverse a string.",
        "What is the capital of France?",
    ]:
        print("  -", prompt)


vLLM 未安装：本环境跳过真实 vLLM 加载，保留下面的完整调用代码结构。
示例 prompts:
  - 请用一句话解释什么是梯度下降。
  - Write a Python function to reverse a string.
  - What is the capital of France?


`LLM` 类的几个关键参数解读：

- `gpu_memory_utilization`：vLLM 占用总显存的比例。设成 0.5 表示只占一半，剩下的留给其他进程。设成 0.9 时 vLLM 会尽可能多用显存做 KV Cache，能并发更多请求，但其他进程就没显存可用了
- `max_model_len`：模型能处理的最大序列长度（输入 + 输出）。这个值直接影响 KV Cache 池大小——值越大能存越多 token，但每个请求占的显存上限也越高
- `dtype`：权重精度。float16 是默认值；为了省显存也可以用 bfloat16（数值范围更大但精度略低）；AWQ/GPTQ 量化模型则用对应的量化 dtype

`SamplingParams` 控制解码行为，和 Notebook 17 讲过的生成策略一致——温度、top-k、top-p 都在这里设置。

## 5. vLLM 启动 OpenAI 兼容服务

离线推理适合脚本和批处理，但生产场景下更常用的是启动一个 HTTP 服务，让前端、后端、其他微服务通过 REST API 调用。vLLM 内置了 `vllm serve` 命令，启动后接口完全兼容 OpenAI 的 Chat Completions API——这意味着所有用 `openai` Python 客户端或 HTTP 调用 OpenAI 的代码，改一下 `base_url` 就能直接用。

启动服务用命令行，不在 notebook 里直接跑，因为服务会一直占用前台。

在终端里执行：

```bash
# 基础启动命令
vllm serve Qwen/Qwen3.5-0.8B \
  --port 8000 \
  --gpu-memory-utilization 0.5 \
  --max-model-len 2048
```

启动成功后会看到类似这样的日志：

```
INFO:     Started server process [12345]
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000
```

此时服务监听 8000 端口，可以接收 OpenAI 格式的请求。`vllm serve` 支持的常用参数：

| 参数 | 作用 |
|:---|:---|
| `--port` | 服务监听端口 |
| `--host` | 监听地址，0.0.0.0 允许外部访问 |
| `--gpu-memory-utilization` | 显存占用比例 |
| `--max-model-len` | 最大序列长度 |
| `--tensor-parallel-size` | 多卡张量并行（大模型用） |
| `--quantization` | 量化方法（awq, gptq, fp8） |
| `--enable-prefix-caching` | 开启前缀缓存（vLLM 版的 RadixAttention） |

In [4]:
# 本 cell 需要先在终端启动 vllm serve，且需要 GPU
# 服务没启动时不让整本 Notebook 失败，而是打印清晰说明。

import requests
import json

VLLM_URL = "http://localhost:8000"

try:
    response = requests.post(
        f"{VLLM_URL}/v1/chat/completions",
        json={
            "model": "Qwen/Qwen3.5-0.8B",
            "messages": [
                {"role": "user", "content": "用一句话解释什么是 transformer。"}
            ],
            "temperature": 0.7,
            "max_tokens": 100,
        },
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    print("状态码:", response.status_code)
    print("回复内容:", data["choices"][0]["message"]["content"])
    print("\n关键元数据：")
    print(f"  完成原因: {data['choices'][0]['finish_reason']}")
    print(f"  prompt tokens: {data['usage']['prompt_tokens']}")
    print(f"  completion tokens: {data['usage']['completion_tokens']}")
except requests.RequestException as e:
    print("vLLM 服务未启动或不可访问；跳过真实 HTTP 调用。")
    print(f"请求地址: {VLLM_URL}/v1/chat/completions")
    print(f"错误类型: {type(e).__name__}")


vLLM 服务未启动或不可访问；跳过真实 HTTP 调用。
请求地址: http://localhost:8000/v1/chat/completions
错误类型: ConnectionError


In [5]:
# 本 cell 同样需要 vllm serve 在运行
# 服务不可访问时打印说明，避免 Notebook 中断。

try:
    response = requests.post(
        f"{VLLM_URL}/v1/chat/completions",
        json={
            "model": "Qwen/Qwen3.5-0.8B",
            "messages": [
                {"role": "user", "content": "写一首关于秋天的五言绝句。"}
            ],
            "stream": True,
            "max_tokens": 80,
        },
        stream=True,
        timeout=10,
    )
    response.raise_for_status()

    print("流式输出：\n")
    for line in response.iter_lines():
        if not line:
            continue
        line = line.decode("utf-8")
        if not line.startswith("data: "):
            continue
        payload = line[6:]
        if payload == "[DONE]":
            break
        chunk = json.loads(payload)
        delta = chunk["choices"][0]["delta"].get("content", "")
        print(delta, end="", flush=True)

    print("\n\n→ 服务通过 SSE 协议逐步推送 token")
except requests.RequestException as e:
    print("vLLM 流式服务未启动或不可访问；跳过真实流式调用。")
    print(f"请求地址: {VLLM_URL}/v1/chat/completions")
    print(f"错误类型: {type(e).__name__}")


vLLM 流式服务未启动或不可访问；跳过真实流式调用。
请求地址: http://localhost:8000/v1/chat/completions
错误类型: ConnectionError


## 6. 用 SGLang 部署同一个模型

SGLang 的使用流程和 vLLM 几乎一致——启动服务，发 OpenAI 兼容请求。差异主要在启动命令和服务端的一些独有优化。

启动 SGLang 服务（同样在终端执行）：

```bash
python -m sglang.launch_server \
  --model-path Qwen/Qwen3.5-0.8B \
  --port 8000 \
  --mem-fraction-static 0.5
```

SGLang 的参数命名和 vLLM 略有不同：

| 参数 | vLLM 对应 | 说明 |
|:---|:---|:---|
| `--model-path` | `--model` | 模型路径 |
| `--port` | `--port` | 服务端口 |
| `--mem-fraction-static` | `--gpu-memory-utilization` | 显存占用比例 |

服务启动后监听同样的 `/v1/chat/completions` 接口。**客户端代码和前面调用 vLLM 的代码一字不差**——把 cell 13 里的 `VLLM_URL` 改成 SGLang 的端口即可，requests 调用、消息结构、流式参数全部通用。这是 OpenAI 兼容协议的最大价值：换推理后端不必改业务代码，业务侧甚至可以做「vLLM 主、SGLang 备」的故障切换。

### SGLang 的结构化输出

SGLang 在结构化输出上做得比 vLLM 更深入。它内置了 constrained decoding，可以在生成时强制输出符合 JSON schema 或正则表达式的文本——这对于让 LLM 输出可解析的结构化数据特别有用。

调用时通过 `json_schema` 参数约束输出格式：

```python
response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen3.5-0.8B",
        "messages": [
            {"role": "user", "content": "介绍 Python 这门编程语言，输出 JSON 格式。"}
        ],
        "extra_body": {
            "json_schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "year": {"type": "integer"},
                    "paradigms": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["name", "year", "paradigms"]
            }
        }
    }
)
```

SGLang 在 decode 阶段会动态屏蔽不符合 schema 的 token，保证输出一定能被 `json.loads` 解析。这比让模型「自己尝试输出 JSON」再事后补救可靠得多。

## 7. vLLM 与 SGLang 选型

把两者放在一张表里对比，便于做技术选型：

| 维度 | vLLM | SGLang |
|:---|:---|:---|
| 核心创新 | PagedAttention（KV Cache 分页） | RadixAttention（前缀共享树） |
| 前缀缓存 | Automatic Prefix Caching（后加） | RadixAttention（原生） |
| 结构化输出 | 通过 outlines/xgrammar 集成 | 原生 constrained decoding |
| 多 LoRA 切换 | 支持得较好 | 支持 |
| 分布式推理 | 支持 TP/PP | 支持 TP |
| 生态成熟度 | 社区更大、issue 更多 | 增长快，论文驱动 |
| 典型场景 | 高并发通用 API | 多轮对话、agent、结构化输出 |

需要说明的是，两个框架的能力在快速收敛——vLLM 加入了 prefix caching 和 structured output，SGLang 也优化了通用吞吐。新版本下选型差异比一年前小得多。具体选哪个，最终要看自己的负载：跑一遍实际 benchmark，比看任何对比表都准。

benchmark 时关注三个指标：吞吐量（tokens/s）、首 token 延迟（TTFT）、尾部延迟（P95/P99）。不同的业务对这三者的敏感度不同——聊天机器人对 TTFT 敏感，离线批处理只看吞吐量。

## 8. 部署自训练模型：核心障碍

前面用 Qwen3.5-0.8B 部署非常顺畅，因为 vLLM 和 SGLang 都内置了 `Qwen3.5 模型架构` 的实现。但如果把自己训练的模型（比如 Notebook 06 里那个 MiniGPT）拿去部署，会直接报错：

```
ValueError: Model architectures ['MiniGPTForCausalLM'] are not supported
on the current platform. Supported architectures: ...
```

这个报错就是核心障碍：推理框架不认识自定义的模型架构。

`config.json` 里的 `architectures` 字段是一个字符串，告诉框架用哪个 Python 类来构造模型。vLLM 和 SGLang 内置了常见架构（LlamaForCausalLM、Qwen3.5 模型架构、MistralForCausalLM 等），但不会预先知道 MiniGPT 长什么样——有多少层、attention 怎么实现、权重名字是什么。

要让框架认识新架构，有两条路径：

- **路径 A：通过 HuggingFace transformers 注册**。把模型包装成 transformers 兼容的 `PreTrainedModel`，再用 `AutoModelForCausalLM.register()` 注册。vLLM 和 SGLang 都会通过 transformers 集成自动发现
- **路径 B：在 vLLM 内部直接注册**。在 vLLM 的 model registry 里用 `@register_model` 装饰器注册一个原生 vLLM 模型实现。这条路径性能更好，但要写更多 vLLM 内部 API

下面两节分别演示这两条路径，然后再处理自定义词表的问题。

## 9. 路径 A：通过 transformers 注册

这是推荐路径：实现一次，vLLM 和 SGLang 都能用，还能直接用 transformers 加载、用 Trainer 训练。

要做三件事：

1. 写一个 `MiniGPTConfig`，继承 `PretrainedConfig`，记录所有架构超参数
2. 写一个 `MiniGPTForCausalLM`，继承 `PreTrainedModel`，实现 forward
3. 用 `AutoConfig.register` 和 `AutoModelForCausalLM.register` 把类注册到 transformers

下面用 Notebook 06 里 MiniGPT 的结构作为例子，演示完整的注册流程。为了在 CPU 上能跑通，hidden_size 和 num_layers 都设小。

In [6]:
# 本 cell 可在 CPU 上运行
# pip install transformers torch

import torch
import torch.nn as nn
from transformers import PretrainedConfig, PreTrainedModel
from transformers import AutoConfig, AutoModelForCausalLM

# 1. 定义 Config 类：记录模型的所有架构超参数
class MiniGPTConfig(PretrainedConfig):
    """MiniGPT 的配置类，记录词表大小、层数、hidden_size 等超参数。"""

    model_type = "minigpt"   # 必须设置，transformers 用这个字段区分不同模型

    def __init__(
        self,
        vocab_size=256,         # 小词表，便于 CPU 演示
        hidden_size=64,         # 小 hidden，便于 CPU 演示
        num_hidden_layers=2,    # 两层就够示意
        num_attention_heads=4,
        intermediate_size=128,  # FFN 中间维度
        max_position_embeddings=128,
        rms_norm_eps=1e-6,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps

print("MiniGPTConfig 定义完成，model_type = 'minigpt'")
print("默认 hidden_size=64, num_layers=2, vocab_size=256")

MiniGPTConfig 定义完成，model_type = 'minigpt'
默认 hidden_size=64, num_layers=2, vocab_size=256


In [7]:
# 2. 定义 Model 类：实现 forward，继承 PreTrainedModel 获得 save/load 能力

class RMSNorm(nn.Module):
    """和 LLaMA/Qwen 一致的 RMSNorm。"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight


class MiniGPTDecoderLayer(nn.Module):
    """简化版 decoder 层：Pre-norm + Self-Attention + 残差，再 Pre-norm + FFN + 残差。"""
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.attn = nn.MultiheadAttention(
            config.hidden_size, config.num_attention_heads, batch_first=True
        )
        self.norm2 = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.ffn = nn.Sequential(
            nn.Linear(config.hidden_size, config.intermediate_size),
            nn.GELU(),
            nn.Linear(config.intermediate_size, config.hidden_size),
        )

    def forward(self, x, attn_mask=None):
        # Pre-norm + Attention + 残差
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x = x + attn_out
        # Pre-norm + FFN + 残差
        h = self.norm2(x)
        x = x + self.ffn(h)
        return x


class MiniGPTForCausalLM(PreTrainedModel):
    """完整的 MiniGPT 语言模型。继承 PreTrainedModel 获得 save_pretrained/load_pretrained。"""

    config_class = MiniGPTConfig   # 告诉 transformers 用哪个 Config 类
    _no_split_modules = ["MiniGPTDecoderLayer"]

    def __init__(self, config):
        super().__init__(config)
        self.config = config

        # token embedding（这里不用 position embedding，留给读者扩展）
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        # decoder 层堆叠
        self.layers = nn.ModuleList([
            MiniGPTDecoderLayer(config) for _ in range(config.num_hidden_layers)
        ])
        # 最终 norm + lm_head
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # PreTrainedModel 要求：初始化权重后调用 post_init
        self.post_init()

    def forward(self, input_ids, attention_mask=None, **kwargs):
        # 1. token ID → embedding
        x = self.embed_tokens(input_ids)
        # 2. 构造因果 mask（下三角）
        T = input_ids.shape[1]
        causal_mask = torch.triu(
            torch.full((T, T), float("-inf"), device=x.device), diagonal=1
        )
        # 3. 逐层
        for layer in self.layers:
            x = layer(x, attn_mask=causal_mask)
        # 4. norm + lm_head → logits
        x = self.norm(x)
        logits = self.lm_head(x)
        return {"logits": logits}


print("MiniGPTForCausalLM 实现完成")
print("结构：embed → 2 × decoder_layer → norm → lm_head")

MiniGPTForCausalLM 实现完成
结构：embed → 2 × decoder_layer → norm → lm_head


In [8]:
# 3. 注册到 transformers，让 AutoConfig 和 AutoModelForCausalLM 能识别

AutoConfig.register("minigpt", MiniGPTConfig)
AutoModelForCausalLM.register(MiniGPTConfig, MiniGPTForCausalLM)

print("注册完成")
print("现在可以用 AutoModelForCausalLM.from_pretrained 加载 MiniGPT 了")

# 验证：构造一个实例并数参数
config = MiniGPTConfig()
model = MiniGPTForCausalLM(config)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n模型参数量: {n_params:,} ({n_params/1e6:.2f}M)")
print("权重名示例：")
for name, _ in list(model.named_parameters())[:5]:
    print(f"  {name}")
print("  ...")

注册完成
现在可以用 AutoModelForCausalLM.from_pretrained 加载 MiniGPT 了

模型参数量: 83,136 (0.08M)
权重名示例：
  embed_tokens.weight
  layers.0.norm1.weight
  layers.0.attn.in_proj_weight
  layers.0.attn.in_proj_bias
  layers.0.attn.out_proj.weight
  ...


In [9]:
# 把 MiniGPT 保存为 HuggingFace 格式
# 保存后整个目录和 Qwen3.5-0.8B 一样可以直接被 vLLM 加载

import os
import json
import tempfile

save_dir = tempfile.mkdtemp(prefix="minigpt-hf-")
model.save_pretrained(save_dir, safe_serialization=False)

print("保存目录内容：")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# 看看 config.json 里有什么——重点看 architectures 字段
print("\nconfig.json 内容：")
with open(os.path.join(save_dir, "config.json")) as fp:
    saved_config = json.load(fp)
print(json.dumps(saved_config, indent=2))

print(f"\n→ architectures 字段自动填成了 ['MiniGPTForCausalLM']")
print(f"→ vLLM 加载时会读这个字段，决定用哪个 Python 类构造模型")

保存目录内容：
  config.json                              0.3 KB
  pytorch_model.bin                        332.1 KB

config.json 内容：
{
  "architectures": [
    "MiniGPTForCausalLM"
  ],
  "dtype": "float32",
  "hidden_size": 64,
  "intermediate_size": 128,
  "max_position_embeddings": 128,
  "model_type": "minigpt",
  "num_attention_heads": 4,
  "num_hidden_layers": 2,
  "rms_norm_eps": 1e-06,
  "transformers_version": "4.57.6",
  "vocab_size": 256
}

→ architectures 字段自动填成了 ['MiniGPTForCausalLM']
→ vLLM 加载时会读这个字段，决定用哪个 Python 类构造模型


### 用 vLLM 加载注册后的 MiniGPT

`config.json` 里的 `architectures` 已经填上 `["MiniGPTForCausalLM"]`。但 vLLM 默认不认识这个名字，需要通过 transformers 集成让 vLLM 发现这个类。

最简单的做法是在 vLLM 启动前，先在 Python 里跑一遍注册代码（前面 cell 23 的 `AutoConfig.register` 和 `AutoModelForCausalLM.register`），vLLM 加载模型时会通过 transformers 的 `AutoModel` 拉起这个类。

写一个 `register_model.py`，在启动 vLLM 之前导入它：

```python
# register_model.py
from transformers import AutoConfig, AutoModelForCausalLM
from minigpt_model import MiniGPTConfig, MiniGPTForCausalLM  # 我们刚写的代码

AutoConfig.register("minigpt", MiniGPTConfig)
AutoModelForCausalLM.register(MiniGPTConfig, MiniGPTForCausalLM)
```

然后在终端启动 vLLM：

```bash
# 通过 --trust-remote-code 让 vLLM 执行用户代码（注册脚本）
vllm serve /tmp/minigpt-hf-xxxxx \
  --port 8000 \
  --enforce-eager
```

`--enforce-eager` 的作用：禁用 CUDA Graph，用 eager 模式执行。MiniGPT 这种自定义类没有为 CUDA Graph 做适配，开启 CUDA Graph 会报错；eager 模式性能稍差但兼容性好。

如果一切顺利，vLLM 日志里会出现 `Model architectures ['MiniGPTForCausalLM']`，服务就跑起来了。客户端代码和前面调用 Qwen3.5-0.8B 完全一样——把 model 字段改成 `MiniGPTForCausalLM` 即可。

需要注意的几个点：

- transformers 路径下的模型走的是 vLLM 的「通用 fallback」实现，性能不如原生优化（PagedAttention 可能用不上）
- 如果模型用了非标准 attention（比如 MLA、Sliding Window），需要额外告诉 vLLM 怎么处理
- 想要最高性能，要走路径 B

## 10. 路径 B：在 vLLM 内部直接注册

路径 A 的限制是性能——transformers 包装的模型走 vLLM 通用执行路径，享受不到 PagedAttention、FlashAttention 等 GPU 优化。如果要把自训练模型部署到高并发生产环境，需要在 vLLM 内部直接写一份实现。

这条路径需要写更多 vLLM 内部 API：继承 `nn.Module`、实现 `forward`、实现 `load_weights`（把 checkpoint 的权重名映射到模型里）、用 `@register_model` 注册。代码量大，但能直接利用 vLLM 的所有优化。

下面是一个骨架，展示了 vLLM 原生模型实现的结构。完整的实现可以参考 vLLM 源码里 `vllm/model_executor/models/llama.py` 这类参考实现。

下面是一个骨架，展示了 vLLM 原生模型实现的结构。完整实现可以参考 vLLM 源码里 `vllm/model_executor/models/llama.py` 这类参考实现。

```python
# 骨架代码：需要装 vllm 且有 GPU 才能实际跑
# from vllm.model_executor.models.registry import register_model
# from vllm.model_executor.layers.sampler import Sampler
# from vllm.model_executor.layers.embedding import VocabParallelEmbedding
# from vllm.model_executor.layers.linear import ParallelLMHead
# from vllm.attention import Attention

@register_model("MiniGPTForCausalLM")
class MiniGPTForCausalLM_vLLM(nn.Module):
    # vLLM 原生实现：直接用 vLLM 的 Attention 层和 Sampler。
    # 关键差异：
    # - 不能用 nn.MultiheadAttention，要用 vllm.attention.Attention
    # - forward 签名变了：接收 positions 和 kv_caches（都由 vLLM 管理）
    # - 必须实现 load_weights，做 checkpoint 名字映射

    def __init__(self, config, *, cache_config=None, quant_config=None):
        super().__init__()
        self.config = config
        # 1. embedding（vLLM 的并行版，支持 TP 切分）
        self.embed_tokens = VocabParallelEmbedding(
            config.vocab_size, config.hidden_size
        )
        # 2. decoder layers（用 vLLM 的 Attention）
        self.layers = nn.ModuleList([
            MiniGPTDecoderLayer_vLLM(config, cache_config, quant_config)
            for _ in range(config.num_hidden_layers)
        ])
        # 3. final norm + lm_head
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size)
        # 4. sampler（vLLM 内置，自动处理 batch）
        self.sampler = Sampler()

    def forward(self, input_ids, positions, kv_caches, **kwargs):
        # vLLM 的 forward 签名：positions 是每个 token 的绝对位置，
        # kv_caches 是每一层对应的 KV Cache（由 vLLM 的 PagedAttention 管理）。
        x = self.embed_tokens(input_ids)
        for i, layer in enumerate(self.layers):
            x = layer(x, positions, kv_caches[i])
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

    def load_weights(self, weights):
        # 从 checkpoint 加载权重。
        # weights 是迭代器，产出 (name, tensor) 对。
        # 这里要做名字映射，把 checkpoint 里的名字对应到模型的 nn.Module 上。
        params_dict = dict(self.named_parameters())
        for name, weight in weights:
            if name in params_dict:
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", default_weight_loader)
                weight_loader(param, weight)
```

核心改动相对路径 A：

- `nn.MultiheadAttention` → `vllm.attention.Attention`
- forward 签名变了：`positions` 和 `kv_caches` 由 vLLM 管理
- 必须实现 `load_weights`，做 checkpoint 名字映射

写完后 vLLM 能用 PagedAttention、FlashAttention 等全部优化。代价是代码量大，每个新模型大约需要 200–400 行。

## 11. 自定义词表

新架构之外，新模型往往也带着新词表。如果训练时用了 Notebook 02 里那个手写的 BPE Tokenizer，vLLM 和 SGLang 是不认识它的——它们只认识 HuggingFace `tokenizers` 库的 `tokenizer.json` 格式。

要让自定义 tokenizer 能被推理框架加载，需要把它转成 HF 格式。最干净的做法是训练时就用 HF `tokenizers` 库训练，直接得到 `tokenizer.json`；如果是已有的自定义格式，需要手动转换。

HF `tokenizers` 库的 BPE 训练 API：

```python
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=["<unk>", "<bos>", "<eos>", "<pad>"]
)
tokenizer.train(files=["corpus.txt"], trainer=trainer)
tokenizer.save("tokenizer.json")
```

保存出来的 `tokenizer.json` 可以直接放进模型目录，vLLM 和 SGLang 加载模型时会自动用它。

如果已经有一个非 HF 格式的 tokenizer（比如 Notebook 02 手写的 BPE），转换思路是：读出 merge rules 和 vocab，按 HF 格式重写一遍。HF 的 BPE 格式有严格的字段要求（`model.vocab`、`model.merges`、`pre_tokenizer`、`decoder` 等），字段不全 vLLM 加载时会报错。

## 12. 端到端：把 MiniGPT 部署起来

把前面的步骤串起来，完整的部署流程：

```text
1. 定义 MiniGPTConfig + MiniGPTForCausalLM（继承 PreTrainedModel）
2. 用 AutoConfig.register / AutoModelForCausalLM.register 注册到 transformers
3. 训练或加载权重
4. model.save_pretrained("minigpt-weights/")
5. tokenizer.save("minigpt-weights/tokenizer.json")
6. 写 register_model.py，跑一次注册
7. vllm serve minigpt-weights/ --trust-remote-code --enforce-eager
8. 用 OpenAI 兼容客户端发请求
```

到第 7 步如果服务能正常启动，第 8 步就能像调用 Qwen3.5-0.8B 一样调用 MiniGPT——客户端代码完全不用改。

最终目录长这样：

```text
minigpt-weights/
├── config.json               # 架构配置，architectures=["MiniGPTForCausalLM"]
├── model.safetensors         # 权重
├── tokenizer.json            # 分词器
├── tokenizer_config.json     # 分词器行为（special tokens 等）
└── generation_config.json    # 默认生成参数
```

这个目录可以打包发给别人，他们 `vllm serve minigpt-weights/` 就能用——前提是他们也注册了 MiniGPT 的模型类。这就是为什么开源模型发布时除了权重，往往还要附带模型类的 Python 代码（比如在 HuggingFace 模型仓库的 `modeling_xxx.py` 文件）。

## 小结

### 推理框架基础（Sections 1-2）

- [ ] HuggingFace transformers 的 static batching 在变长请求场景下吞吐很低
- [ ] continuous batching 让新请求在每一步 decode 时都能加入或退出 batch
- [ ] PagedAttention 把 KV Cache 分页管理，减少显存碎片（vLLM 提出）
- [ ] RadixAttention 把共享前缀的 KV Cache 复用（SGLang 提出）

### 部署现成模型（Sections 3-7）

- [ ] vLLM 的两种使用方式：`LLM` 离线推理和 `vllm serve` 启动 HTTP 服务
- [ ] vLLM 的服务接口完全兼容 OpenAI Chat Completions API
- [ ] SGLang 的使用流程和 vLLM 几乎一致，客户端代码可以无修改复用
- [ ] 关键参数：`gpu_memory_utilization`、`max_model_len`、`dtype`
- [ ] 流式输出通过 SSE 协议逐 token 推送
- [ ] 选型：纯吞吐优先 vLLM，结构化输出和共享前缀优先 SGLang

### 部署自训练模型（Sections 8-12）

- [ ] 核心障碍：推理框架不认识新架构（`config.json` 的 `architectures` 字段）
- [ ] 路径 A：通过 transformers 注册（简单，性能一般，推荐入门）
- [ ] 路径 B：在 vLLM 内部用 `@register_model` 注册（复杂，性能最优）
- [ ] 自定义词表要转成 HF `tokenizers` 的 `tokenizer.json` 格式
- [ ] 端到端流程：定义类 → 注册 → 训练 → save_pretrained → vllm serve

### 一句话总结

部署现成模型是工程问题，部署自训练模型是「如何让推理框架认识你的代码」的问题——前者的难点在调参和压测，后者的难点在适配 transformers 与 vLLM 的接口约定。

### 参考资料

- vLLM 文档：https://docs.vllm.ai/
- vLLM GitHub：https://github.com/vllm-project/vllm
- vLLM 添加新模型：https://docs.vllm.ai/en/latest/models/adding_model.html
- SGLang 文档：https://docs.sglang.ai/
- SGLang GitHub：https://github.com/sgl-project/sglang
- PagedAttention 论文（SOSP 2023）：https://arxiv.org/abs/2309.06180
- SGLang 论文：https://arxiv.org/abs/2312.07104
- Orca 论文（continuous batching 来源，OSDI 2022）：https://www.usenix.org/conference/osdi22/presentation/yu
- HuggingFace tokenizers 文档：https://huggingface.co/docs/tokenizers
- HuggingFace 自定义模型教程：https://huggingface.co/docs/transformers/custom_models
- Qwen3 模型卡：https://huggingface.co/Qwen/Qwen3.5-0.8B

## 作业

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

### 作业 1：continuous batching 的吞吐优势

阅读 cell 3 的 static batching 与 continuous batching 模拟代码。当 `batch_size` 增大到 16、而请求数还是 8 个时，static batching 的总耗时是变大、变小还是不变？为什么？

小提示：注意 `simulate_static_batching` 里 `batch_time = max(batch)` 这行——一个 batch 的耗时由谁决定。

```python
# 作业 1：填 True 或 False
answer_1 = None  # True 表示「会变」，False 表示「不变」

# 你的理由：
reason_1 = ""

assert answer_1 is not None, "请填 True 或 False"
assert isinstance(answer_1, bool), "请填 True 或 False"
if answer_1 is False:
    print("✅ 作业 1 通过")
    print("   batch_size=16 但只有 8 个请求 → 只有 1 个 batch")
    print("   batch_time = max(全部 8 个请求的长度)")
    print("   结果和 batch_size=4 时分两批的总和相同（max(前4)+max(后4)）")
    print("   实际上还要看具体长度分布，但这里的关键是：")
    print("   batch_size 超过请求数时，多余的空间不会带来加速")
else:
    print("提示：注意 batch_size 超过请求数时，batch 的数量怎么变")
```

### 作业 2：vLLM 关键参数对显存的影响

`LLM` 类有三个关键参数影响显存：`gpu_memory_utilization`、`max_model_len`、`dtype`。假设 GPU 总显存 24GB、模型权重占 4GB，下面哪种配置能让 vLLM 的 KV Cache 池最大？

```python
# 作业 2：选 A / B / C / D
answer_2 = ""

# A. gpu_memory_utilization=0.5, max_model_len=2048, dtype="float16"
# B. gpu_memory_utilization=0.9, max_model_len=8192, dtype="float16"
# C. gpu_memory_utilization=0.9, max_model_len=2048, dtype="float16"
# D. gpu_memory_utilization=0.9, max_model_len=2048, dtype="float32"

assert not answer_2.startswith("在这里"), "请填 A/B/C/D"
assert answer_2 in "ABCD", "请填 A/B/C/D 中的一个字母"

if answer_2 == "C":
    print("✅ 作业 2 通过")
    print("   KV Cache 池 ≈ gpu_memory_utilization × 总显存 - 权重大小")
    print("   C 的池 = 0.9 × 24 - 4 = 17.6 GB，是最大的")
    print("   D 用了 float32，权重翻倍到 8GB，反而缩小了池子")
    print("   B 的 max_model_len 大但池子大小不变（只是每个请求的上限更高）")
else:
    print(f"你选了 {answer_2}")
    print("提示：max_model_len 决定的是单请求上限，不是池子大小")
```

### 作业 3：自定义架构的注册顺序

如果想让 vLLM 加载一个全新的模型架构 `MyAwesomeLLM`，下面步骤应该按什么顺序排列？

```python
# 作业 3：把字母按正确顺序填入
answer_3 = ""

# A. vllm serve my-model/
# B. 实现 MyAwesomeLLMForCausalLM(PreTrainedModel) 类
# C. model.save_pretrained("my-model/")
# D. AutoModelForCausalLM.register(MyAwesomeLLMConfig, MyAwesomeLLMForCausalLM)
# E. 实现 MyAwesomeLLMConfig(PretrainedConfig) 类
# F. 训练或加载权重

assert len(answer_3) == 6, "请按正确顺序填入 6 个字母"
assert set(answer_3) == set("ABCDEF"), "字母不能重复或遗漏"

if answer_3 == "EBFDC A".replace(" ", ""):
    print("✅ 作业 3 通过")
    print("   顺序：E（Config）→ B（Model）→ F（权重）→ D（注册）→ C（保存）→ A（启动）")
    print("   关键：先有 Config 才能定义 Model；Model 注册后才能正确保存和加载")
else:
    print(f"你的顺序：{answer_3}")
    print("提示：先定义 Config，再实现 Model；先注册，再保存和启动服务")
```